# Supervised CNN + OpenMax — Open-Set NIDS

**Why this approach:**
- Supervised CNN trained on labeled known classes — learns class boundaries directly
- Keras `model.fit()` handles mixed precision loss scaling automatically — no NaN
- Class weights handle the 80% Normal imbalance
- OpenMax on penultimate layer for unknown detection (same as InfoGAN approach)
- Trains in **10-30 minutes** total, not hours

**Same setup as InfoGAN notebook:** attach `nids-helper-scripts` + `chethuhn/network-intrusion-dataset`

In [ ]:
import sys, os

HELPERS_DIR = "/kaggle/input/nids-helper-scripts"
sys.path.insert(0, HELPERS_DIR)

import infogan_model_kaggle as igo
import nids_utils_kaggle as nu
import openmax_kaggle as om

# GPU + mixed precision
igo.setup_kaggle()

import numpy as np
import pandas as pd
import json, joblib, time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════
ALL_CSV_PATHS = [nu.MONDAY_CSV] + nu.ATTACK_FILES
SAVE_DIR = "/kaggle/working/cnn_openmax"
os.makedirs(SAVE_DIR, exist_ok=True)

KNOWN_CLASSES = [0, 1, 2, 3, 5, 6]
UNKNOWN_CLASSES = [4]
NUM_KNOWN = len(KNOWN_CLASSES)

BATCH_SIZE = 512
EPOCHS = 50
LR = 0.001

TAIL_SIZE = 200
ALPHA_RANK = 5
DISTANCE_TYPE = "cosine"

known_class_names = [igo.CLASS_NAMES[c] for c in KNOWN_CLASSES]
print(f"Known classes ({NUM_KNOWN}): {known_class_names}")
print(f"Unknown: {[igo.CLASS_NAMES[c] for c in UNKNOWN_CLASSES]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SCALER
# ═══════════════════════════════════════════════════════════════
SCALER_PATH = os.path.join(nu.PREPROCESSING_DIR, "scaler_infogan_kaggle.pkl")
FEATURE_COLS_PATH = os.path.join(nu.PREPROCESSING_DIR, "infogan_kaggle_feature_cols.json")

if os.path.exists(SCALER_PATH) and os.path.exists(FEATURE_COLS_PATH):
    print("Loading existing scaler...")
    scaler = joblib.load(SCALER_PATH)
    with open(FEATURE_COLS_PATH) as f:
        feature_cols = json.load(f)
else:
    print("Fitting scaler...")
    scaler, feature_cols = igo.fit_scaler_streaming(ALL_CSV_PATHS, feature_range=(-1, 1))
    os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
    joblib.dump(scaler, SCALER_PATH)
    with open(FEATURE_COLS_PATH, "w") as f:
        json.dump(feature_cols, f)
print(f"Features: {len(feature_cols)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREPROCESS + BUILD DATASETS (with labels, train/val split)
# ═══════════════════════════════════════════════════════════════
import glob as _glob
old_npy = _glob.glob(os.path.join(nu.PREPROCESSING_DIR, "*.npy"))
if old_npy:
    print(f"Cleaning {len(old_npy)} cached .npy files...")
    for f in old_npy:
        os.remove(f)

images_path, labels_path, total_samples = igo.preprocess_csvs_to_numpy(
    ALL_CSV_PATHS, scaler, feature_cols,
    known_classes=KNOWN_CLASSES, save_dir=nu.PREPROCESSING_DIR,
)

from sklearn.model_selection import train_test_split

images = np.load(images_path)
labels = np.load(labels_path)
print(f"Total: {len(images):,} samples")

X_train, X_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.15, random_state=42, stratify=labels
)
del images, labels

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .cache().shuffle(50000)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CLASS WEIGHTS (handle 80% Normal imbalance)
# ═══════════════════════════════════════════════════════════════
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight("balanced", classes=np.arange(NUM_KNOWN), y=y_train)
class_weight_dict = dict(enumerate(cw))
print("Class weights:")
for i, name in enumerate(known_class_names):
    print(f"  {name}: {class_weight_dict[i]:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# BUILD CNN CLASSIFIER
# ═══════════════════════════════════════════════════════════════
def build_nids_classifier(num_classes):
    inp = layers.Input(shape=(11, 11, 1), name="image_input")

    # Same conv architecture as InfoGAN backbone
    x = layers.Conv2D(64, 4, strides=2, padding="same",
                      kernel_initializer="he_normal")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Conv2D(128, 4, strides=2, padding="same",
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Conv2D(256, 4, strides=1, padding="same",
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    x = layers.Flatten()(x)

    # Penultimate layer — 128-dim (used for OpenMax)
    x = layers.Dense(128, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.2, name="penultimate")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation="softmax",
                       dtype="float32", name="class_output")(x)

    return keras.Model(inp, out, name="nids_classifier")

model = build_nids_classifier(NUM_KNOWN)
model.summary()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPILE + TRAIN
# Keras handles mixed precision loss scaling automatically!
# ═══════════════════════════════════════════════════════════════
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, "best_model.keras"),
        monitor="val_accuracy", save_best_only=True, mode="max", verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1,
    ),
]

print(f"Training for up to {EPOCHS} epochs (early stopping patience=10)...")
t0 = time.time()

history_obj = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

train_time = time.time() - t0
history = history_obj.history
print(f"\nTraining done in {train_time/60:.1f} min")
print(f"Best val_accuracy: {max(history['val_accuracy']):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# VERIFICATION — check training is healthy
# ═══════════════════════════════════════════════════════════════
final_val_acc = history["val_accuracy"][-1]
final_val_loss = history["val_loss"][-1]
best_val_acc = max(history["val_accuracy"])

print(f"Final val_accuracy: {final_val_acc:.4f}")
print(f"Best val_accuracy:  {best_val_acc:.4f}")
print(f"Final val_loss:     {final_val_loss:.4f}")

if best_val_acc < 0.5:
    print("\nWARNING: Val accuracy below 50% — model may not have learned properly.")
elif best_val_acc < 0.8:
    print("\nModel learned something but accuracy is moderate.")
else:
    print("\nModel training looks healthy.")

# Loss/accuracy plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(history["accuracy"], label="train")
axes[1].plot(history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
fig.suptitle("CNN Classifier Training")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LOAD EVALUATION DATA
# ═══════════════════════════════════════════════════════════════
# Use the val split as test_known (already done above)
X_test_known = X_val
y_test_known = y_val

# For OpenMax fitting, use training data
X_train_eval = X_train
y_train_eval = y_train

# Unknown data
unk_img_path = os.path.join(nu.PREPROCESSING_DIR, "unknown_images.npy")
if os.path.exists(unk_img_path):
    X_test_unknown = np.load(unk_img_path)
else:
    print("Loading unknown data from CSVs...")
    X_test_unknown, _ = igo.build_image_label_arrays_streamed(
        ALL_CSV_PATHS, scaler, feature_cols, known_classes=UNKNOWN_CLASSES
    )
    np.save(unk_img_path, X_test_unknown)

print(f"Train (OpenMax fit): {len(X_train_eval):,}")
print(f"Test (known):        {len(X_test_known):,}")
print(f"Test (unknown):      {len(X_test_unknown):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CLASSIFIER EVALUATION (direct, no Hungarian matching needed!)
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import classification_report, accuracy_score

y_pred_test = np.argmax(model.predict(X_test_known, batch_size=1024, verbose=0), axis=1)

cls_acc = accuracy_score(y_test_known, y_pred_test)
print(f"CNN Classifier Test Accuracy: {cls_acc:.4f}")
print(classification_report(
    y_test_known, y_pred_test,
    target_names=known_class_names, zero_division=0
))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FIT OPENMAX — penultimate layer (128-dim) + true labels
# ═══════════════════════════════════════════════════════════════
# Extract penultimate layer activations
penultimate_model = keras.Model(
    model.input, model.get_layer("penultimate").output
)

print("Extracting activation vectors (train)...")
avs_train = penultimate_model.predict(X_train_eval, batch_size=1024, verbose=0)
print(f"Activation vectors: {avs_train.shape}")

# TRUE labels for MAV computation
mavs, class_avs = om.compute_mavs(avs_train, y_train_eval, NUM_KNOWN)
print(f"MAVs computed for {len(mavs)} classes")

distances = om.compute_distances(class_avs, mavs, distance_type=DISTANCE_TYPE)
weibull_params = om.fit_weibull(distances, tail_size=TAIL_SIZE)

weibull_ok = 0
for cls, params in weibull_params.items():
    name = known_class_names[cls]
    if params:
        print(f"  {name}: shape={params[0]:.4f}, scale={params[2]:.6f}")
        weibull_ok += 1
    else:
        print(f"  {name}: FAILED")
print(f"\nWeibull fit: {weibull_ok}/{len(weibull_params)} succeeded")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPENMAX PREDICTION
# ═══════════════════════════════════════════════════════════════
print("Extracting activation vectors (test)...")
avs_test_known = penultimate_model.predict(X_test_known, batch_size=1024, verbose=0)

t0 = time.time()
preds_known, probs_known = om.openmax_predict(
    avs_test_known, mavs, weibull_params,
    alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
    distance_type=DISTANCE_TYPE,
)
print(f"OpenMax on {len(avs_test_known):,} known: {time.time()-t0:.1f}s")

known_correct = np.sum((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_as_unknown = np.sum(preds_known == NUM_KNOWN)
print(f"Correctly classified: {known_correct:,} / {len(y_test_known):,}")
print(f"False unknowns:      {known_as_unknown:,}")

if len(X_test_unknown) > 0:
    avs_test_unknown = penultimate_model.predict(X_test_unknown, batch_size=1024, verbose=0)
    preds_unknown, probs_unknown = om.openmax_predict(
        avs_test_unknown, mavs, weibull_params,
        alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
        distance_type=DISTANCE_TYPE,
    )
    unk_det = np.sum(preds_unknown == NUM_KNOWN)
    print(f"Unknown detected: {unk_det}/{len(preds_unknown)} ({unk_det/len(preds_unknown):.4f})")
else:
    preds_unknown = np.array([], dtype=int)
    probs_unknown = np.empty((0, NUM_KNOWN + 1))
    print("No unknown test data.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPREHENSIVE METRICS
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import (
    precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    multilabel_confusion_matrix,
)
from sklearn.preprocessing import label_binarize

all_class_names = known_class_names + (["Unknown"] if len(X_test_unknown) > 0 else [])
num_all = NUM_KNOWN + (1 if len(X_test_unknown) > 0 else 0)

if len(X_test_unknown) > 0:
    y_true_all = np.concatenate([y_test_known, np.full(len(X_test_unknown), NUM_KNOWN)])
    y_pred_all = np.concatenate([preds_known, preds_unknown])
    probs_all = np.concatenate([probs_known, probs_unknown])
else:
    y_true_all = y_test_known
    y_pred_all = preds_known
    probs_all = probs_known

prec, rec, f1, support = precision_recall_fscore_support(
    y_true_all, y_pred_all, labels=list(range(num_all)), zero_division=0
)
mcm = multilabel_confusion_matrix(y_true_all, y_pred_all, labels=list(range(num_all)))
fpr_list = []
for i in range(num_all):
    tn, fp, fn, tp = mcm[i].ravel()
    fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)

metrics_df = pd.DataFrame({
    "Class": all_class_names, "Precision": prec, "Recall": rec,
    "F1-Score": f1, "FPR": fpr_list, "Support": support.astype(int),
})
metrics_df = pd.concat([metrics_df, pd.DataFrame([{
    "Class": "Macro Avg", "Precision": prec.mean(), "Recall": rec.mean(),
    "F1-Score": f1.mean(), "FPR": np.mean(fpr_list), "Support": support.sum(),
}, {
    "Class": "Weighted Avg",
    "Precision": np.average(prec, weights=np.maximum(support, 1)),
    "Recall": np.average(rec, weights=np.maximum(support, 1)),
    "F1-Score": np.average(f1, weights=np.maximum(support, 1)),
    "FPR": np.average(fpr_list, weights=np.maximum(support, 1)),
    "Support": support.sum(),
}])], ignore_index=True)

print("=" * 80)
print("  CNN + OPENMAX — PERFORMANCE METRICS")
print("=" * 80)
print(metrics_df.to_string(index=False, float_format="%.4f"))
print(f"\nOverall Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ROC + PR CURVES
# ═══════════════════════════════════════════════════════════════
y_true_bin = label_binarize(y_true_all, classes=list(range(num_all)))
if y_true_bin.shape[1] == 1:
    y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

colors = plt.cm.tab10(np.linspace(0, 1, num_all))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
auc_scores = {}
ax = axes[0]
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_scores[all_class_names[i]] = auc_i
    ax.plot(fpr_i, tpr_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} ({auc_i:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC (OvR)")
ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
all_fpr = np.linspace(0, 1, 200)
mean_tpr = np.zeros_like(all_fpr)
valid = 0
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i); valid += 1
mean_tpr /= max(valid, 1)
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "navy", lw=2, label=f"Macro ({macro_auc:.3f})")
ax.fill_between(all_fpr, 0, mean_tpr, alpha=0.1, color="navy")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("Macro-Avg ROC")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "roc_curves.png"), dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ap_scores = {}
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    prec_i, rec_i, _ = precision_recall_curve(y_true_bin[:, i], probs_all[:, i])
    ap_i = average_precision_score(y_true_bin[:, i], probs_all[:, i])
    ap_scores[all_class_names[i]] = ap_i
    ax.plot(rec_i, prec_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} ({ap_i:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall"); ax.legend(loc="lower left", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "pr_curves.png"), dpi=150)
plt.show()

print("\nAUC per class:")
for c, a in auc_scores.items(): print(f"  {c}: {a:.4f}")
print(f"  Macro: {macro_auc:.4f}")
print("\nAvg Precision:")
for c, a in ap_scores.items(): print(f"  {c}: {a:.4f}")
print(f"  mAP: {np.mean(list(ap_scores.values())):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPEN-SET METRICS + CONFUSION MATRIX
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("  OPEN-SET RECOGNITION METRICS")
print("=" * 60)

known_acc = np.mean((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_rejection = np.mean(preds_known == NUM_KNOWN)
print(f"Known Classification Accuracy: {known_acc:.4f}")
print(f"Known Rejection Rate:          {known_rejection:.4f}")

if len(X_test_unknown) > 0:
    udr = np.mean(preds_unknown == NUM_KNOWN)
    print(f"Unknown Detection Rate:        {udr:.4f}")
    print(f"False Known Rate:              {1 - udr:.4f}")
    h_score = 2 * known_acc * udr / (known_acc + udr) if (known_acc + udr) > 0 else 0
    print(f"H-Score (KCA & UDR harmonic):  {h_score:.4f}")
else:
    udr = 0.0
    h_score = 0.0

cm = confusion_matrix(y_true_all, y_pred_all)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ticks = np.arange(num_all)
ax.set_xticks(ticks); ax.set_xticklabels(all_class_names, rotation=45, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(all_class_names)
fig.colorbar(im); fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE ALL ARTIFACTS
# ═══════════════════════════════════════════════════════════════
model.save(os.path.join(SAVE_DIR, "nids_classifier.keras"))

openmax_data = {
    "mavs": {str(k): v.tolist() for k, v in mavs.items()},
    "weibull_params": {str(k): list(v) if v else None for k, v in weibull_params.items()},
    "known_classes": KNOWN_CLASSES, "unknown_classes": UNKNOWN_CLASSES,
    "tail_size": TAIL_SIZE, "alpha_rank": ALPHA_RANK,
    "distance_type": DISTANCE_TYPE, "num_known": NUM_KNOWN,
}
with open(os.path.join(SAVE_DIR, "openmax_params.json"), "w") as f:
    json.dump(openmax_data, f, indent=2)

np.savez(os.path.join(SAVE_DIR, "openmax_mavs.npz"),
         **{f"class_{k}": v for k, v in mavs.items()})

metrics_df.to_csv(os.path.join(SAVE_DIR, "performance_metrics.csv"), index=False)

pd.DataFrame({
    "Class": list(auc_scores.keys()), "AUC": list(auc_scores.values()),
    "AP": [ap_scores.get(c, np.nan) for c in auc_scores.keys()]
}).to_csv(os.path.join(SAVE_DIR, "auc_ap_scores.csv"), index=False)

meta = {
    "model_type": "cnn_openmax",
    "approach": "supervised_cnn_classifier_with_openmax",
    "num_known": NUM_KNOWN,
    "known_classes": KNOWN_CLASSES, "unknown_classes": UNKNOWN_CLASSES,
    "known_class_names": known_class_names,
    "epochs_trained": len(history["loss"]),
    "batch_size": BATCH_SIZE, "lr": LR,
    "tail_size": TAIL_SIZE, "alpha_rank": ALPHA_RANK,
    "distance_type": DISTANCE_TYPE,
    "train_time_min": train_time / 60,
    "best_val_accuracy": float(best_val_acc),
    "classifier_test_accuracy": float(cls_acc),
    "overall_accuracy": float(accuracy_score(y_true_all, y_pred_all)),
    "macro_auc": float(macro_auc),
    "h_score": float(h_score),
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"All artifacts saved -> {SAVE_DIR}")
for fname in sorted(os.listdir(SAVE_DIR)):
    if not os.path.isdir(os.path.join(SAVE_DIR, fname)):
        print(f"  {fname}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# HYPERPARAMETER SWEEP
# ═══════════════════════════════════════════════════════════════
if len(X_test_unknown) > 0:
    results = []
    for ts in [20, 50, 100, 200, 500]:
        wp = om.fit_weibull(distances, tail_size=ts)
        for ar in [1, 2, 3, 5, 7, 10]:
            pk, _ = om.openmax_predict(
                avs_test_known, mavs, wp, ar, NUM_KNOWN, DISTANCE_TYPE
            )
            pu, _ = om.openmax_predict(
                avs_test_unknown, mavs, wp, ar, NUM_KNOWN, DISTANCE_TYPE
            )
            ka = np.mean((pk != NUM_KNOWN) & (pk == y_test_known))
            ud = np.mean(pu == NUM_KNOWN)
            hs = 2 * ka * ud / (ka + ud) if (ka + ud) > 0 else 0
            results.append({
                "tail_size": ts, "alpha_rank": ar,
                "known_acc": round(ka, 4), "unknown_det": round(ud, 4),
                "h_score": round(hs, 4),
            })
    sweep_df = pd.DataFrame(results).sort_values("h_score", ascending=False)
    sweep_df.to_csv(os.path.join(SAVE_DIR, "hyperparameter_sweep.csv"), index=False)
    print("Top 10 configs by H-Score:")
    print(sweep_df.head(10).to_string(index=False))
    best = sweep_df.iloc[0]
    print(f"\nBest: tail={int(best.tail_size)}, alpha={int(best.alpha_rank)}, H={best.h_score:.4f}")
else:
    print("No unknown data — skipping sweep.")

## Why CNN+OpenMax vs InfoGAN+OpenMax

| Aspect | InfoGAN+OpenMax | CNN+OpenMax |
|--------|----------------|-------------|
| Training | Unsupervised (adversarial) | Supervised (labeled) |
| Stability | GAN training is unstable, prone to NaN/mode collapse | Standard backprop, very stable |
| Class learning | Must discover classes via mutual information | Directly learns class boundaries |
| Training time | Hours (200 epochs) | Minutes (10-30 epochs) |
| Mixed precision | Manual loss scaling needed | Keras handles automatically |
| Open-set detection | OpenMax on Q classifier penultimate layer | OpenMax on CNN penultimate layer |
| Expected accuracy | 60-70% (unsupervised discovery) | 90%+ (supervised) |